# Grating coupler size sweep

Coupling efficiency and channel capacity of a multifunctional grating coupler as a function
of coupler size, by 3D FDTD in Tidy3D.

## Contents

1. [Configuration](#1-configuration)
2. [Credentials](#2-credentials)
3. [Slab waveguide mode](#3-slab-waveguide-mode)
4. [Design](#4-design)
5. [Simulation construction](#5-simulation-construction)
6. [Job submission](#6-job-submission)
7. [Coupling efficiency](#7-coupling-efficiency)
8. [Channel analysis](#8-channel-analysis)
9. [Sweep](#9-sweep)
10. [Results](#10-results)

## 1. Configuration

In [ ]:
import datetime
import json
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy import optimize

import tidy3d as td
import tidy3d.web as web

In [ ]:
WL = 1.55
N_CORE = 3.0
N_CLAD = 1.0
T_WG = 0.22
ETCH_DEPTH = 0.07

POINTS_PER_WL = 20
RUN_TIME = 12e-13
GC_TO_SRC = 0.5 * WL
SRC_TO_PML = 0.5 * WL
SIM_SIZE_Z = 2 * WL
MONITOR_DIST = 0.5 * WL

N_FOURIER = 4
TARGET_NA = 0.2
ANGLE_OVERSAMPLE = 1.5

CE_CUT = 0.05
RAD_PER_LOBE = 6.0
ANG_PER_LOBE = 4.0

SIZES_TO_RUN = [5.0, 7.5, 10.0, 12.5, 15.0, 17.5]
UNPATTERNED_ANGLES = "all"
BUDGET_FC = 45.0
DRY_RUN = True
KEEP_RAW_DATA = False
FOLDER_NAME = "gc_size_sweep"
OUTDIR = "gc_size_sweep_outputs"

In [ ]:
OMEGA = 2 * np.pi / WL
FREQ = td.C_0 / WL
FWIDTH = 0.5 * (td.C_0 / (WL - 5e-2) - td.C_0 / (WL + 5e-2))
CORE = td.Medium(permittivity=N_CORE ** 2)
CHI_MAX = N_CORE ** 2 - 1
os.makedirs(OUTDIR, exist_ok=True)

## 2. Credentials

Delete this section before publishing. The key may also be left empty and supplied through `~/.config/tidy3d/config.toml`.

In [ ]:
API_KEY = ""

if API_KEY:
    web.configure(API_KEY)
print(web.account())

## 3. Slab waveguide mode

In [ ]:
def slab_te_params(k, thickness, n_core, n_clad, m=0):
    a = thickness / 2
    v = k * a * np.sqrt(n_core ** 2 - n_clad ** 2)
    disp = lambda b: v * np.sqrt(1 - b) - m * np.pi / 2 - np.arctan(np.sqrt(b / (1 - b)))
    b = optimize.root_scalar(disp, x0=0.05, x1=0.95, method="secant").root
    beta = np.sqrt(b * (n_core ** 2 - n_clad ** 2) + n_clad ** 2) * k
    return beta, v * np.sqrt(1 - b) / a, v * np.sqrt(b) / a, m * np.pi / 2


BETA, KAPPA, XI, PHI = slab_te_params(OMEGA, T_WG, N_CORE, N_CLAD)
N_EFF = BETA / OMEGA
THETA_MAX = np.arcsin(OMEGA / BETA)

In [ ]:
def _decay(z, sign):
    return np.exp(-XI * np.maximum(sign * z - T_WG / 2, 0.0))


def mode_e_perp(z):
    z = np.asarray(z, float)
    return np.where(
        z > T_WG / 2, np.cos(KAPPA * T_WG / 2 - PHI) * _decay(z, 1),
        np.where(z < -T_WG / 2, np.cos(KAPPA * T_WG / 2 + PHI) * _decay(z, -1),
                 np.cos(KAPPA * z - PHI)))


def mode_h_perp(z):
    return (BETA / OMEGA) * mode_e_perp(z)


def mode_h_par(z):
    z = np.asarray(z, float)
    return 1j / OMEGA * np.where(
        z > T_WG / 2, -XI * np.cos(KAPPA * T_WG / 2 - PHI) * _decay(z, 1),
        np.where(z < -T_WG / 2, XI * np.cos(KAPPA * T_WG / 2 + PHI) * _decay(z, -1),
                 KAPPA * np.sin(KAPPA * z - PHI)))


_zn = np.linspace(-T_WG / 2 - 2 * WL, T_WG / 2 + 2 * WL, 1000)
MODE_NORM = 1.0 / np.sqrt(
    0.5 * abs(np.sum(mode_e_perp(_zn) * np.conj(mode_h_perp(_zn))) * (_zn[1] - _zn[0])))

print(f"beta = {BETA:.4f} /um   n_eff = {N_EFF:.4f}   theta_max = {np.degrees(THETA_MAX):.2f} deg")

## 4. Design

In [ ]:
def grating_vectors():
    angles = np.arange(N_FOURIER) * np.pi / N_FOURIER
    q = BETA - TARGET_NA * OMEGA
    return q * np.cos(angles), -q * np.sin(angles)


def design_permittivity(length):
    pixel = WL / POINTS_PER_WL
    x = np.arange(-length / 2, length / 2, pixel)
    y = np.arange(-length / 2, length / 2, pixel)
    z = np.arange(T_WG / 2 - ETCH_DEPTH, T_WG / 2, pixel)
    qx, qy = grating_vectors()
    xx, yy = np.meshgrid(x, y, indexing="ij")
    modulation = np.cos(qx * xx[..., None] + qy * yy[..., None]).sum(axis=-1)
    eps2d = 1 + CHI_MAX * (N_FOURIER + modulation) / (2 * N_FOURIER)
    return x, y, z, np.repeat(eps2d[:, :, None], len(z), axis=2)

## 5. Simulation construction

In [ ]:
def simulation_size(length):
    return length + 2 * (GC_TO_SRC + SRC_TO_PML)


def angle_list(length):
    step = ANGLE_OVERSAMPLE * 2 * np.pi / (BETA * length)
    return step * np.arange(int(np.floor(THETA_MAX / step)))


def port_weights(n_theta):
    return np.r_[8.0, np.full(n_theta - 1, 16.0)]

In [ ]:
def _tfsf_sources(length, theta, box):
    pos = box / 2 - SRC_TO_PML
    tang = np.linspace(-pos, pos, round(length / WL * POINTS_PER_WL))
    zs = np.linspace(-SIM_SIZE_Z / 2, SIM_SIZE_Z / 2, round(SIM_SIZE_Z / WL * POINTS_PER_WL))
    e_perp = mode_e_perp(zs) * MODE_NORM
    h_perp = mode_h_perp(zs) * MODE_NORM
    h_par = mode_h_par(zs) * MODE_NORM
    ct, st = np.cos(theta), np.sin(theta)
    pulse = td.GaussianPulse(freq0=FREQ, fwidth=FWIDTH)

    sources = []
    for axis, s in (("x", -1), ("x", 1), ("y", -1), ("y", 1)):
        xs, ys = (np.array([s * pos]), tang) if axis == "x" else (tang, np.array([s * pos]))
        phase = (np.exp(1j * BETA * ct * xs)[:, None] * np.exp(-1j * BETA * st * ys)[None, :])
        prof = lambda a: phase[:, :, None, None] * a[None, None, :, None]
        if axis == "x":
            comps = {"Ey": s * prof(h_perp) / td.ETA_0,
                     "Ez": s * st * prof(h_par) / td.ETA_0,
                     "Hz": s * ct * prof(e_perp)}
        else:
            comps = {"Ex": -s * prof(h_perp) / td.ETA_0,
                     "Ez": s * ct * prof(h_par) / td.ETA_0,
                     "Hz": -s * st * prof(e_perp)}
        coords = dict(x=xs, y=ys, z=zs, f=[FREQ])
        dataset = td.FieldDataset(
            **{k: td.ScalarFieldDataArray(v, coords=coords) for k, v in comps.items()})
        sources.append(td.CustomCurrentSource(
            source_time=pulse,
            center=(xs.mean(), ys.mean(), zs.mean()),
            size=(np.ptp(xs), np.ptp(ys), np.ptp(zs)),
            current_dataset=dataset))
    return sources


def _side_monitors(length):
    half = length / 2
    faces = (("left", (0, length, td.inf), (-half, 0, 0)),
             ("right", (0, length, td.inf), (half, 0, 0)),
             ("bot", (length, 0, td.inf), (0, -half, 0)),
             ("top", (length, 0, td.inf), (0, half, 0)))
    monitors = []
    for name, size, center in faces:
        monitors.append(td.FieldMonitor(size=size, center=center, freqs=[FREQ],
                                        name=f"field_{name}"))
        monitors.append(td.FluxMonitor(size=size, center=center, freqs=[FREQ],
                                       name=f"flux_{name}"))
    return monitors


def _radiation_monitors():
    z = T_WG / 2 + MONITOR_DIST
    plane = (td.inf, td.inf, 0)
    return [td.FieldMonitor(size=plane, center=(0, 0, z), freqs=[FREQ], name="field_up"),
            td.FieldMonitor(size=plane, center=(0, 0, -z), freqs=[FREQ], name="field_down"),
            td.FluxMonitor(size=plane, center=(0, 0, z), freqs=[FREQ], name="flux_up"),
            td.FluxMonitor(size=plane, center=(0, 0, -z), freqs=[FREQ], name="flux_down")]

In [ ]:
def make_sim_unpatterned(length, theta):
    box = simulation_size(length)
    slab = td.Structure(
        geometry=td.Box.from_bounds(rmin=(-1e3, -1e3, -T_WG / 2), rmax=(1e3, 1e3, T_WG / 2)),
        medium=CORE)
    return td.Simulation(
        center=(0, 0, 0),
        size=(box, box, SIM_SIZE_Z),
        grid_spec=td.GridSpec.auto(min_steps_per_wvl=POINTS_PER_WL, wavelength=WL),
        structures=[slab],
        sources=_tfsf_sources(length, theta, box),
        monitors=_side_monitors(length),
        run_time=RUN_TIME,
        boundary_spec=td.BoundarySpec(x=td.Boundary.pml(), y=td.Boundary.pml(),
                                      z=td.Boundary.pml()),
        symmetry=(0, 0, 0))


def make_sim_patterned(length, theta):
    sim = make_sim_unpatterned(length, theta)
    x, y, z, eps = design_permittivity(length)
    medium = td.CustomMedium(
        permittivity=td.SpatialDataArray(eps, coords=dict(x=x, y=y, z=z)))
    pattern = td.Structure(
        geometry=td.Box(size=(length, length, ETCH_DEPTH),
                        center=(0, 0, T_WG / 2 - ETCH_DEPTH / 2)),
        medium=medium)
    return sim.updated_copy(structures=list(sim.structures) + [pattern],
                            monitors=list(sim.monitors) + _radiation_monitors())

## 6. Job submission

In [ ]:
def task_table(size_in_wl):
    length = size_in_wl * WL
    thetas = angle_list(length)
    unpat = {"ends": sorted({0, len(thetas) - 1}), "all": range(len(thetas)),
             "first": [0]}[UNPATTERNED_ANGLES]
    tasks = {f"L{size_in_wl:g}_unpat_{i}": ("unpat", i, thetas[i]) for i in unpat}
    tasks.update({f"L{size_in_wl:g}_pat_{i}": ("pat", i, t) for i, t in enumerate(thetas)})
    return length, thetas, tasks


def build_batch(size_in_wl):
    length, thetas, tasks = task_table(size_in_wl)
    sims = {name: (make_sim_unpatterned(length, t) if kind == "unpat"
                   else make_sim_patterned(length, t))
            for name, (kind, _, t) in tasks.items()}
    return web.Batch(simulations=sims, folder_name=FOLDER_NAME, verbose=True), tasks


def estimate(batch):
    total = 0.0
    for name, job in batch.jobs.items():
        cost = web.estimate_cost(job.task_id, verbose=False)
        total += cost
        print(f"  {name:<24} {cost:8.3f} FC")
    print(f"  {'TOTAL':<24} {total:8.3f} FC")
    return total


def submit(size_in_wl):
    path = os.path.join(OUTDIR, f"L_{size_in_wl:g}")
    os.makedirs(path, exist_ok=True)
    batch_file = os.path.join(path, "batch.json")
    if os.path.exists(batch_file):
        return web.Batch.from_file(batch_file), task_table(size_in_wl)[2], path

    batch, tasks = build_batch(size_in_wl)
    total = estimate(batch)
    if total > BUDGET_FC:
        raise RuntimeError(f"estimated {total:.3f} FC exceeds BUDGET_FC = {BUDGET_FC}")
    batch.to_file(batch_file)
    batch.run(path_dir=path)
    return batch, tasks, path


def run_parameters(size_in_wl, batch, tasks):
    length, thetas, _ = task_table(size_in_wl)
    qx, qy = grating_vectors()
    n_rad, n_ang = polar_resolution(length)
    dx, dy, dz, eps = design_permittivity(length)
    probe = make_sim_patterned(length, thetas[-1])
    return {
        "timestamp": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
        "tidy3d_version": td.__version__,
        "medium": {"wl": WL, "n_core": N_CORE, "n_clad": N_CLAD, "t_wg": T_WG,
                   "etch_depth": ETCH_DEPTH},
        "mode": {"beta": BETA, "n_eff": N_EFF, "kappa": KAPPA, "xi": XI, "phi": PHI,
                 "theta_max": THETA_MAX, "norm": MODE_NORM},
        "design": {"n_fourier": N_FOURIER, "target_na": TARGET_NA, "chi_max": CHI_MAX,
                   "qx": qx.tolist(), "qy": qy.tolist(),
                   "pixel": WL / POINTS_PER_WL, "shape": list(eps.shape),
                   "eps_min": float(eps.min()), "eps_max": float(eps.max())},
        "geometry": {"size_in_wl": size_in_wl, "length": length,
                     "simulation_size": simulation_size(length),
                     "sim_size_z": SIM_SIZE_Z, "gc_to_src": GC_TO_SRC,
                     "src_to_pml": SRC_TO_PML, "monitor_dist": MONITOR_DIST},
        "discretisation": {"points_per_wl": POINTS_PER_WL, "run_time": RUN_TIME,
                           "num_cells": int(probe.num_cells),
                           "num_time_steps": int(probe.num_time_steps),
                           "shutoff": probe.shutoff},
        "angles": {"oversample": ANGLE_OVERSAMPLE,
                   "step": ANGLE_OVERSAMPLE * 2 * np.pi / (BETA * length),
                   "theta": thetas.tolist(),
                   "theta_deg": np.degrees(thetas).tolist(),
                   "n_theta": len(thetas),
                   "n_ports": int(port_weights(len(thetas)).sum()),
                   "unpatterned_angles": UNPATTERNED_ANGLES},
        "analysis": {"ce_cut": CE_CUT, "rad_per_lobe": RAD_PER_LOBE,
                     "ang_per_lobe": ANG_PER_LOBE, "n_rad": n_rad, "n_ang": n_ang},
        "tasks": {name: {"task_id": batch.jobs[name].task_id, "kind": kind, "index": index,
                         "theta": theta,
                         "estimated_cost": web.estimate_cost(batch.jobs[name].task_id,
                                                             verbose=False),
                         "real_cost": web.real_cost(batch.jobs[name].task_id, verbose=False)}
                  for name, (kind, index, theta) in tasks.items()},
    }

## 7. Coupling efficiency

In [ ]:
def analytic_input_power(length, theta):
    return length * (np.cos(theta) + np.sin(theta))


def flux_table(batch_data, tasks, length):
    rows = {}
    for name, (kind, index, theta) in tasks.items():
        data = batch_data[name]
        row = rows.setdefault(index, {"theta": theta})
        if kind == "unpat":
            row["p_in"] = float(abs(data["flux_left"].flux.values[0])
                                + abs(data["flux_top"].flux.values[0]))
        else:
            row["p_up"] = float(abs(data["flux_up"].flux.values[0]))
            row["p_down"] = float(abs(data["flux_down"].flux.values[0]))
            row["p_guided_out"] = float(abs(data["flux_right"].flux.values[0])
                                        + abs(data["flux_bot"].flux.values[0]))
    for row in rows.values():
        row["launch_calibration"] = row["p_in"] / analytic_input_power(length, row["theta"])
    return [rows[i] for i in sorted(rows)]


def coupling_efficiencies(rows):
    for row in rows:
        row["ce_up"] = row["p_up"] / row["p_in"]
        row["ce_down"] = row["p_down"] / row["p_in"]
        row["accounted"] = (row["p_up"] + row["p_down"] + row["p_guided_out"]) / row["p_in"]
    return rows


def save_fields(batch_data, tasks, path):
    for name, (kind, index, _) in tasks.items():
        if kind != "pat":
            continue
        for side in ("up", "down"):
            data = batch_data[name][f"field_{side}"]
            with h5py.File(os.path.join(path, f"field_{side}_{index}.h5"), "w") as fh:
                for comp in ("Ex", "Ey", "Hx", "Hy"):
                    fh.create_dataset(
                        f"dataset_{comp}",
                        data=getattr(data, comp).to_numpy()[:, :, 0, 0].astype(np.complex64))
                for axis in ("x", "y", "z"):
                    fh.create_dataset(f"dataset_{axis}", data=data.Ex[axis].to_numpy())


def discard_raw_data(path):
    for name in os.listdir(path):
        if name.startswith("fdve-") or name == "batch.hdf5":
            os.remove(os.path.join(path, name))

## 8. Channel analysis

The radiated spectrum is evaluated directly from the monitor coordinates on a polar grid whose angular sampling is a multiple of eight, so every rotation and mirror of the eightfold symmetry group is an exact index permutation.

In [ ]:
def cell_widths(u):
    w = np.empty_like(u)
    w[1:-1] = 0.5 * (u[2:] - u[:-2])
    w[0] = u[1] - u[0]
    w[-1] = u[-1] - u[-2]
    return w


def polar_resolution(length):
    n_rad = max(32, int(np.ceil(RAD_PER_LOBE * length / WL)))
    n_ang = max(64, 8 * int(np.ceil(ANG_PER_LOBE * OMEGA * length / 8)))
    return n_rad, n_ang


def polar_kgrid(n_rad, n_ang, kmax):
    nodes, weights = np.polynomial.legendre.leggauss(n_rad)
    kr = 0.5 * kmax * (nodes + 1.0)
    wr = 0.5 * kmax * weights
    phi = 2 * np.pi * np.arange(n_ang) / n_ang
    return (np.outer(kr, np.cos(phi)), np.outer(kr, np.sin(phi)),
            np.outer(kr * wr, np.full(n_ang, 2 * np.pi / n_ang)))


def spectrum(field, x, y, kx, ky, chunk=8192):
    weighted = (field * cell_widths(x)[:, None] * cell_widths(y)[None, :]).astype(np.complex64)
    out = np.empty(kx.size, np.complex128)
    for i in range(0, kx.size, chunk):
        sl = slice(i, min(i + chunk, kx.size))
        wy = np.exp(-1j * np.outer(y, ky[sl])).astype(np.complex64)
        wx = np.exp(-1j * np.outer(x, kx[sl])).astype(np.complex64)
        out[sl] = ((weighted @ wy) * wx).sum(axis=0)
    return out / (2 * np.pi)


def real_space_power(field, x, y):
    return float((np.abs(field) ** 2
                  * cell_widths(x)[:, None] * cell_widths(y)[None, :]).sum())


def validate_transform(x, y, kx, ky, w):
    xx, yy = np.meshgrid(x, y, indexing="ij")
    taper = lambda u: np.sin(np.pi * (u - u[0]) / (u[-1] - u[0])) ** 2
    window = np.outer(taper(x), taper(y))
    worst = 0.0
    for k0x, k0y in ((1.0, 0.5), (-2.2, 1.3), (0.3, -3.0)):
        f = window * np.exp(1j * (k0x * xx + k0y * yy))
        s = spectrum(f, x, y, kx.ravel(), ky.ravel())
        worst = max(worst, abs(float(((np.abs(s) ** 2) * w.ravel()).sum())
                               / real_space_power(f, x, y) - 1.0))
    return worst


def symmetry_images(spec, wsq, n_ang, skip_mirror):
    m = np.arange(n_ang)
    rows, labels = [], []
    for j in range(8):
        rows.append((spec[:, (m - j * n_ang // 8) % n_ang] * wsq).ravel())
        labels.append((j, 0))
        if not skip_mirror:
            rows.append((spec[:, (n_ang // 2 - m + j * n_ang // 8) % n_ang] * wsq).ravel())
            labels.append((j, 1))
    return rows, labels

In [ ]:
def channel_matrix(path, rows, length):
    n_rad, n_ang = polar_resolution(length)
    kx, ky, w = polar_kgrid(n_rad, n_ang, OMEGA)
    wsq = np.sqrt(w)
    matrix, diagnostics, labels = [], [], []
    for index, row in enumerate(rows):
        with h5py.File(os.path.join(path, f"field_down_{index}.h5"), "r") as fh:
            ey = fh["/dataset_Ey"][:].astype(np.complex128)
            x = fh["/dataset_x"][:]
            y = fh["/dataset_y"][:]
        x = x - 0.5 * (x[0] + x[-1])
        y = y - 0.5 * (y[0] + y[-1])
        if index == 0:
            err = validate_transform(x, y, kx, ky, w)
            print(f"  transform validation: {err:.2e}")
            assert err < 5e-3
        ey *= np.sqrt(row["ce_down"] / real_space_power(ey, x, y))
        spec = spectrum(ey, x, y, kx.ravel(), ky.ravel()).reshape(kx.shape)
        images, image_labels = symmetry_images(spec, wsq, n_ang, index == 0)
        norms = np.array([float(np.vdot(v, v).real) for v in images])
        diagnostics.append({"index": index, "row_norm": norms.mean(),
                            "rotation_spread": float(np.ptp(norms) / norms.mean()),
                            "light_cone_fraction": norms.mean() / row["ce_down"]})
        matrix.extend(images)
        labels.extend([(index, a, b) for a, b in image_labels])
    return np.array(matrix), diagnostics, np.array(labels)


def channel_efficiencies(matrix):
    ce = np.sort(np.linalg.eigvalsh(matrix @ matrix.conj().T).real)[::-1]
    return np.clip(ce, 0.0, None)


def summarise(size_in_wl, rows, ce):
    weights = port_weights(len(rows))
    n_coupled = int((ce >= CE_CUT).sum())
    return {
        "size_in_wl": size_in_wl,
        "n_theta": len(rows),
        "n_ports": int(weights.sum()),
        "mean_ce_down_ports": float((weights * [r["ce_down"] for r in rows]).sum()
                                    / weights.sum()),
        "mean_ce_up_ports": float((weights * [r["ce_up"] for r in rows]).sum()
                                  / weights.sum()),
        "n_coupled": n_coupled,
        "mean_ce_channels": float(ce[:n_coupled].mean()) if n_coupled else 0.0,
        "throughput": float(ce[:n_coupled].sum()),
        "throughput_all_channels": float(ce.sum()),
        "best_channel_ce": float(ce[0]),
        "launch_calibration": float(np.mean([r["launch_calibration"] for r in rows])),
        "energy_accounted": float(np.mean([r["accounted"] for r in rows])),
    }

## 9. Sweep

In [ ]:
def run_size(size_in_wl):
    batch, tasks, path = submit(size_in_wl)
    length = size_in_wl * WL
    batch_data = batch.load(path_dir=path)
    save_fields(batch_data, tasks, path)

    rows = coupling_efficiencies(flux_table(batch_data, tasks, length))
    matrix, diagnostics, labels = channel_matrix(path, rows, length)
    ce = channel_efficiencies(matrix)
    result = summarise(size_in_wl, rows, ce)

    parameters = run_parameters(size_in_wl, batch, tasks)
    result["real_cost"] = sum(t["real_cost"] or 0.0 for t in parameters["tasks"].values())
    np.savez(os.path.join(path, "channels.npz"), ce=ce,
             row_norms=np.linalg.norm(matrix, axis=1), labels=labels,
             port_weight=port_weights(len(rows)), omega=OMEGA, ce_cut=CE_CUT)
    with open(os.path.join(path, "parameters.json"), "w") as fh:
        json.dump(parameters, fh, indent=2)
    with open(os.path.join(path, "summary.json"), "w") as fh:
        json.dump({"result": result, "rows": rows, "diagnostics": diagnostics}, fh, indent=2)
    if not KEEP_RAW_DATA:
        discard_raw_data(path)
    return result, rows, ce, diagnostics

In [ ]:
results = {}
if DRY_RUN:
    for size in SIZES_TO_RUN:
        print(f"=== L = {size:g} wl (dry run) ===")
        dry_batch, _ = build_batch(size)
        estimate(dry_batch)
        dry_batch.delete()
else:
    for size in SIZES_TO_RUN:
        print(f"=== L = {size:g} wl ===")
        results[size] = run_size(size)

## 10. Results

In [ ]:
for size, (result, rows, ce, diagnostics) in results.items():
    print(f"L = {size:g} wl")
    for key, value in result.items():
        print(f"  {key:<26} {value}")
    print(f"  {'theta (deg)':>12} {'CE_down':>9} {'CE_up':>9} {'P_in':>12} "
          f"{'accounted':>10} {'light cone':>11}")
    for row, diag in zip(rows, diagnostics):
        print(f"  {np.degrees(row['theta']):12.4f} {row['ce_down']:9.4%} {row['ce_up']:9.4%} "
              f"{row['p_in']:12.6e} {row['accounted']:10.4f} "
              f"{diag['light_cone_fraction']:11.4f}")

In [ ]:
for size, (result, _, ce, _) in results.items():
    fig, ax = plt.subplots(figsize=(7.2, 5.4))
    ax.scatter(np.arange(1, len(ce) + 1), 100 * ce, s=42, c="red", zorder=3)
    ax.axhline(100 * CE_CUT, color="0.35", lw=1.6, ls="--", zorder=2)
    ax.annotate(f"{100 * CE_CUT:.0f}%  ({result['n_coupled']} channels)",
                xy=(len(ce), 100 * CE_CUT), xytext=(-8, 6), textcoords="offset points",
                ha="right", fontsize=15, color="0.25")
    ax.set_xlim(0, len(ce) + max(2, len(ce) // 20))
    ax.set_ylim(0, 10 * np.ceil(100 * ce[0] / 10) + 5)
    ax.set_xlabel("singular value number", fontsize=19)
    ax.set_ylabel("coupling efficiency (%)", fontsize=19)
    ax.set_title(f"$L = {size:g}\\lambda$", fontsize=19)
    ax.tick_params(labelsize=17)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, f"L_{size:g}", "channels.png"), dpi=220)
    plt.show()